# Week 3 Context and State Practice

Compare run-scoped context with mutable, checkpointed conversation state.

In [1]:
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from pprint import pprint
from typing import Any

from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / '.venv' / 'Scripts' / 'python.exe').exists():
    project_root = project_root.parent
load_dotenv(project_root / '.env')
assert Path(sys.executable).resolve() == (project_root / '.venv' / 'Scripts' / 'python.exe').resolve()
assert sys.version_info[:3] == (3, 13, 1)
model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')
context_question = 'What is my name, how long is my presentation, and who is my audience?'
context_system = 'Answer using only information available in the current conversation or through available tools. If the user has not provided requested information, say that you do not know instead of guessing.'

def show_messages(label: str, messages: list[Any]) -> None:
    print(f'\n===== {label} =====')
    for index, message in enumerate(messages, start=1):
        if isinstance(message, HumanMessage):
            role = 'HUMAN MESSAGE'
        elif isinstance(message, AIMessage) and message.tool_calls:
            role = 'AI TOOL CALL'
        elif isinstance(message, ToolMessage):
            role = 'TOOL RESULT'
        elif isinstance(message, AIMessage):
            role = 'AI ANSWER'
        else:
            role = type(message).__name__.upper()
        print(f'\n--- Message {index}: {role} ---')
        pprint(message)

print('Using project .venv, Python 3.13.1, and Gemini Flash-Lite.')

Using project .venv, Python 3.13.1, and Gemini Flash-Lite.


## Experiment 1: Context is not automatically visible

The schema values are passed as runtime context, not in the prompt. This agent has no tool that reads them.

In [2]:
@dataclass
class PresentationContext:
    user_name: str = 'Abdelmalek'
    presentation_minutes: int = 8
    audience: str = 'mixed technical and non-technical'

context_without_tool_agent = create_agent(
    model=model,
    system_prompt=context_system,
    context_schema=PresentationContext,
)
context_without_tool_result = context_without_tool_agent.invoke(
    {'messages': [HumanMessage(content=context_question)]},
    context=PresentationContext(),
)
context_without_tool_answer = context_without_tool_result['messages'][-1]
show_messages('Without a context-reading tool', context_without_tool_result['messages'])
print('Context supplied to invocation but absent from messages/tools: True')
print('Answer content:')
pprint(context_without_tool_answer.content)


===== Without a context-reading tool =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='What is my name, how long is my presentation, and who is my audience?', additional_kwargs={}, response_metadata={}, id='a5b61439-8af4-451b-a0d4-ea436474d071')

--- Message 2: AI ANSWER ---
AIMessage(content=[{'type': 'text', 'text': 'I do not know your name, the length of your presentation, or who your audience is, as you have not provided that information in this conversation.', 'extras': {'signature': 'EnMKcQFpFH0TZ7b1yTOmihlV94TCS28hyV32Xx4D3PZA15efAfZGK4mGgM+bBn2FSzFf6hTXETfc7STjOmvgqHnaGYUcehCGJfoeMug861p8rIeXvxrmXCKOqzkwBxGWdXlR1iyK0t2CRHc4iqOipphyDCF1'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fea7-7d1a-7f41-b6d3-d09b50a6e807-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens': 30, 'total_tokens': 81, 

## Experiment 2: Read context through a tool

In [3]:
@tool
def get_presentation_context(runtime: ToolRuntime[PresentationContext]) -> dict[str, Any]:
    '''Read the run-scoped user, presentation duration, and audience context.'''
    return asdict(runtime.context)

context_agent = create_agent(
    model=model,
    tools=[get_presentation_context],
    system_prompt=context_system,
    context_schema=PresentationContext,
)
context_tool_result = context_agent.invoke(
    {'messages': [HumanMessage(content=context_question)]},
    context=PresentationContext(),
)
show_messages('With a context-reading tool', context_tool_result['messages'])
context_answer_text = str(context_tool_result['messages'][-1].content)
context_details_present = all(
    value in context_answer_text.lower()
    for value in ('abdelmalek', '8', 'mixed', 'technical', 'non-technical')
)
print('Answer reflects the three context fields:', context_details_present)


===== With a context-reading tool =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='What is my name, how long is my presentation, and who is my audience?', additional_kwargs={}, response_metadata={}, id='446514a1-86d3-4592-ae4e-2e662452532f')

--- Message 2: AI TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_presentation_context', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_418308': 'EnMKcQFpFH0TCnf4Tl8cvlRl/KSEK3HsibQl1LzkONVizhl1sgezx1ozKQKQteyQ37jONeIA0ew0nkWy7h+tjlapFHe7GuZgsAQGXJujZCf/UTCvU0yO2IAIX7IqsrvKdAZXlbWKfJRxmK9IG//8SXihmdw4'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fea7-9490-7941-a209-a46cee2240bf-0', tool_calls=[{'name': 'get_presentation_context', 'args': {}, 'id': 'call_418308', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 'output_tokens

In [4]:
override_result = context_agent.invoke(
    {'messages': [HumanMessage(content=context_question)]},
    context=PresentationContext(presentation_minutes=12),
)
show_messages('Context override: presentation_minutes=12', override_result['messages'])
override_answer_text = str(override_result['messages'][-1].content)
print('Override reflected as 12 minutes:', '12' in override_answer_text)
print('Schema default remains:', PresentationContext().presentation_minutes)


===== Context override: presentation_minutes=12 =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='What is my name, how long is my presentation, and who is my audience?', additional_kwargs={}, response_metadata={}, id='411b080f-c88c-4a8d-9212-e9deea1d2f9a')

--- Message 2: AI TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_presentation_context', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_521021': 'EnMKcQFpFH0T4D/i6yT8NY9r4TGpRBXlKtOJLruwZJuFQO4kZt6xtVMoGJJ6hJLnbDe2K4QpOFbmLua+8tVS2TRmur99f8kZ9TUZT1DFUnFhhK6JIhd/Z3jIkGqZbod+uJbOZkGiP7KrjC3+QTHtyVCy1LHK'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fea7-b145-7533-8c9d-1b88dd3bc776-0', tool_calls=[{'name': 'get_presentation_context', 'args': {}, 'id': 'call_521021', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 

## Experiment 3: Learn and update custom state

The initial value is written once when the thread starts. Later invocations use the same thread and do not reinitialize it.

In [5]:
class RehearsalState(AgentState):
    preferred_explanation_style: str


@tool
def read_preferred_explanation_style(runtime: ToolRuntime[RehearsalState]) -> str:
    '''Read the preferred explanation style stored in this conversation state.'''
    return runtime.state.get('preferred_explanation_style', 'not specified')


@tool
def update_preferred_explanation_style(
    preferred_explanation_style: str,
    runtime: ToolRuntime[RehearsalState],
) -> Command:
    '''Save the user's preferred explanation style to conversation state.'''
    return Command(update={
        'preferred_explanation_style': preferred_explanation_style,
        'messages': [ToolMessage(
            content='Updated the preferred explanation style.',
            tool_call_id=runtime.tool_call_id,
        )],
    })


state_checkpointer = InMemorySaver()
state_agent = create_agent(
    model=model,
    tools=[read_preferred_explanation_style, update_preferred_explanation_style],
    system_prompt=(
        'Use read_preferred_explanation_style to answer questions about the saved preference. '
        'When the user explicitly asks to save a preference, use update_preferred_explanation_style. '
        'Treat the tool result as the saved value and do not claim an update unless the tool succeeds.'
    ),
    state_schema=RehearsalState,
    checkpointer=state_checkpointer,
)
state_config = {'configurable': {'thread_id': 'context-state-session-1'}}
initial_state_result = state_agent.invoke(
    {
        'messages': [HumanMessage(content='What explanation style have I chosen? Use the available tool to check.')],
        'preferred_explanation_style': 'not specified',
    },
    config=state_config,
)
show_messages('Initial preference lookup', initial_state_result['messages'])
state_before_update = state_agent.get_state(state_config).values
print('Checkpointed preference before update:', state_before_update.get('preferred_explanation_style'))
pprint(state_before_update)


===== Initial preference lookup =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='What explanation style have I chosen? Use the available tool to check.', additional_kwargs={}, response_metadata={}, id='c2ede0c4-e64d-42fb-8cbd-403011d37a2d')

--- Message 2: AI TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_preferred_explanation_style', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_472426': 'EnMKcQFpFH0TW69luRTWpA8thOK/CB0I8qhVW0OVHdd5Sarb0GsbK/zYyw3Wvdspw1cfPl1+n05s63qF6kn617lpvrohYApxhTc4X6IGYW8maP5kRQD0FfQJgG6q4wOA6qSlA1RzgKij9XsrkF28GS8CigzA'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fea7-caca-7c23-8eee-777e6aa9e821-0', tool_calls=[{'name': 'read_preferred_explanation_style', 'args': {}, 'id': 'call_472426', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158

In [6]:
update_result = state_agent.invoke(
    {'messages': [HumanMessage(content='For my rehearsals, I prefer short analogies followed by one technical sentence. Save this as my preferred explanation style.')]},
    config=state_config,
)
show_messages('Preference update', update_result['messages'])
state_after_update = state_agent.get_state(state_config).values
print('Checkpointed preference after update:', state_after_update.get('preferred_explanation_style'))
pprint(state_after_update)


===== Preference update =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='What explanation style have I chosen? Use the available tool to check.', additional_kwargs={}, response_metadata={}, id='c2ede0c4-e64d-42fb-8cbd-403011d37a2d')

--- Message 2: AI TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_preferred_explanation_style', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_472426': 'EnMKcQFpFH0TW69luRTWpA8thOK/CB0I8qhVW0OVHdd5Sarb0GsbK/zYyw3Wvdspw1cfPl1+n05s63qF6kn617lpvrohYApxhTc4X6IGYW8maP5kRQD0FfQJgG6q4wOA6qSlA1RzgKij9XsrkF28GS8CigzA'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fea7-caca-7c23-8eee-777e6aa9e821-0', tool_calls=[{'name': 'read_preferred_explanation_style', 'args': {}, 'id': 'call_472426', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'outpu

In [7]:
follow_up_result = state_agent.invoke(
    {'messages': [HumanMessage(content='What explanation style have I chosen? Use the available tool to check.')]},
    config=state_config,
)
show_messages('Follow-up preference lookup on the same thread', follow_up_result['messages'])
final_checkpoint_state = state_agent.get_state(state_config).values
print('Persisted preference retrieved:', final_checkpoint_state.get('preferred_explanation_style'))
print('State value is present after follow-up:', final_checkpoint_state.get('preferred_explanation_style') == 'short analogies followed by one technical sentence')


===== Follow-up preference lookup on the same thread =====

--- Message 1: HUMAN MESSAGE ---
HumanMessage(content='What explanation style have I chosen? Use the available tool to check.', additional_kwargs={}, response_metadata={}, id='c2ede0c4-e64d-42fb-8cbd-403011d37a2d')

--- Message 2: AI TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_preferred_explanation_style', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_472426': 'EnMKcQFpFH0TW69luRTWpA8thOK/CB0I8qhVW0OVHdd5Sarb0GsbK/zYyw3Wvdspw1cfPl1+n05s63qF6kn617lpvrohYApxhTc4X6IGYW8maP5kRQD0FfQJgG6q4wOA6qSlA1RzgKij9XsrkF28GS8CigzA'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fea7-caca-7c23-8eee-777e6aa9e821-0', tool_calls=[{'name': 'read_preferred_explanation_style', 'args': {}, 'id': 'call_472426', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata

## Classifying agent information

| Runtime context | Mutable state |
|---|---|
| User identity for the current run | Preferred explanation style learned during conversation |
| Presentation duration supplied for the current session | Current rehearsal outline |
| Audience supplied for the current session | Sources reviewed and their verification status |

This classification depends on application design. In this exercise, duration and audience are fixed inputs for each run, while learned preferences and working notes can be updated and checkpointed.

## Observations

- **Before adding the context-reading tool**  
  The agent said it did not know my name, presentation duration or audience. Defining and supplying runtime context did not automatically include those details in the model’s messages.

- **After adding the context-reading tool**  
  The agent called the tool and correctly retrieved Abdelmalek, an eight-minute presentation and a mixed technical/non-technical audience.

- **Overriding the context**  
  Passing a presentation duration of 12 minutes made the answer reflect that value. The schema’s default remained eight minutes.

- **State before and after learning the preference**  
  The checkpointed preferred_explanation_style field changed from “not specified” to “short analogies followed by one technical sentence.”

- **Persistence in the same thread**  
  A later invocation retrieved the saved preference through the state-reading tool. This confirmed that the custom field persisted across turns in the same thread.

- **Why context and state should not be confused**  
  Runtime context provides application-supplied information for a run. State stores information that can evolve during the conversation. In this application, identity, duration and audience were context, while the learned explanation preference was mutable state. InMemorySaver retains that state only while the current process remains running.